# brats-uad — Colab training (Drive-persistent)
Dataset → fast local disk; **all outputs autosave to Drive**. Disconnects are harmless — re-run cells 1–3, then training, and `--resume` continues.
Full guide: `docs/COLAB.md`. **Pick a T4 GPU** (Runtime ▸ Change runtime type).


### 1. Mount Drive + check GPU


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv


### 2. Clone the code


In [ ]:
!git clone https://github.com/<you>/brats-uad.git /content/brats-uad
%cd /content/brats-uad


### 3. One-shot setup — installs deps, extracts dataset to local SSD, makes Drive dirs, audits
Pass your packed tar name. Outputs go to `MyDrive/brats-uad/`.


In [ ]:
!bash scripts/colab_bootstrap.sh brats_dev.tar /content/drive/MyDrive/brats-uad


### 4. Train — every cell sources the env file written by step 3 (`--resume` survives disconnects)


In [ ]:
!source /content/brats_env.sh && python src/train_vae.py --epochs 25 --bs 16 --num-workers 2 --resume
!source /content/brats_env.sh && python src/compute_scaling_factor.py --n 2000
!source /content/brats_env.sh && python src/train_diffusion.py --epochs 20 --bs 32 --num-workers 2 --resume


### 5. TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/brats-uad/logs


### 6. Calibrate, evaluate, XAI


In [ ]:
!source /content/brats_env.sh && python src/calibrate.py
!source /content/brats_env.sh && python src/evaluate.py --n-images 60 --max-plots 16
!source /content/brats_env.sh && python src/make_report_figures.py
!source /content/brats_env.sh && python src/make_xai_panels.py --n 8


### On reconnect
Re-run cells 1–3 (extract is skipped if already present), then cell 4 — training continues from `last.pt`.
